In [20]:
import pandas as pd, numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA
from catboost import CatBoostRegressor
import xgboost as xgb
# 1. Chargement des données (Utilise tes noms de fichiers sauvegardés)
X_train = pd.read_csv("../features/global_features/X_train.csv")
y_train = pd.read_csv("../features/global_features/y_train.csv")
X_test = pd.read_csv("../features/global_features/X_test.csv")

In [21]:
import umap
from sklearn.preprocessing import StandardScaler


from sklearn.impute import SimpleImputer

# On définit l'imputer
imputer = SimpleImputer(strategy='mean')

# On l'applique sur X_train et X_test
X_train_clean = imputer.fit_transform(X_train.drop(columns=['video_id'], errors='ignore'))
X_test_clean = imputer.transform(X_test.drop(columns=['video_id'], errors='ignore'))

# 1. Standardisation (Indispensable pour UMAP)
scaler_umap = StandardScaler()
# On peut l'appliquer sur le X_train complet (888 features) ou le filtré
X_train_scaled = scaler_umap.fit_transform(X_train_clean)
X_test_scaled = scaler_umap.transform(X_test_clean)

# 2. Configuration de UMAP
# n_neighbors : équilibre entre structure locale (bas) et globale (haut)
# min_dist : contrôle la densité des amas (0.1 est une valeur standard)
reducer = umap.UMAP(
    n_neighbors=15,
    n_components=20, 
    min_dist=0.1,
    metric='euclidean',
    random_state=42
)

# 3. Fit sur le train et transformation des deux

X_train_umap = reducer.fit_transform(X_train_scaled)
X_test_umap = reducer.transform(X_test_scaled)

# 4. Conversion en DataFrame pour ton modèle LightGBM/CatBoost
X_train_umap_df = pd.DataFrame(X_train_umap, columns=[f'umap_{i}' for i in range(20)], index=X_train.index)
X_test_umap_df = pd.DataFrame(X_test_umap, columns=[f'umap_{i}' for i in range(20)], index=X_test.index)

print(f"Forme finale : {X_train_umap_df.shape}")

/home/thibault/.local/miniconda3/envs/hackaton/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Forme finale : (1348, 20)


In [22]:
X_train_umap_df = pd.concat([X_train.iloc[:,0],X_train_umap_df],axis=1)
X_test_umap_df = pd.concat([X_test.iloc[:,0],X_test_umap_df],axis=1)

In [23]:
X_train_umap_df.to_csv("../features/global_features/X_train_umap.csv", index=False)
X_test_umap_df.to_csv("../features/global_features/X_test_umap.csv", index=False)